# Week 6: Building a Baseline Triage Model

This notebook trains and evaluates three baselines on the Mercer/Yale ED triage dataset: a stratified random baseline, logistic regression, and a bounded decision tree. The emphasis is clinical defensibility, not sophistication.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_SEED = 42
TEST_SIZE = 0.20
TREE_MAX_DEPTH = 5
ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_PATH = ROOT / 'data' / 'processed' / 'week5_triage_cleaned.parquet'
print(DATA_PATH)

In [ ]:
df = pd.read_parquet(DATA_PATH)
df['esi_numeric'] = pd.to_numeric(df['esi'].astype(str), errors='coerce')
df = df[df['esi_numeric'].isin([1,2,3,4,5])].copy()
df['esi_numeric'] = df['esi_numeric'].astype(int)
print(df.shape)
df['esi_numeric'].value_counts(normalize=True).sort_index().mul(100).round(2)

## Feature Set

For this first baseline, I avoided race/ethnicity and downstream laboratory/imaging fields. The model uses early triage information: age, arrival mode, previous use, triage vital signs, time context, and common/urgent chief complaints.

In [ ]:
if 'arrivalmode_ambulance_or_helicopter' not in df.columns:
    arrival = df['arrivalmode'].astype(str).str.lower()
    df['arrivalmode_ambulance_or_helicopter'] = arrival.str.contains('ambulance|helicopter|ems|critical', regex=True).astype(float)

base_numeric = ['age','n_edvisits','n_admissions','triage_vital_hr','triage_vital_sbp','triage_vital_dbp','triage_vital_rr','triage_vital_o2','triage_vital_temp','arrivalmode_ambulance_or_helicopter']
base_categorical = ['gender','arrivalmode','arrivalmonth','arrivalday','arrivalhour_bin','previousdispo']
cc_features = ['cc_abdominalpain','cc_other','cc_chestpain','cc_shortnessofbreath','cc_backpain','cc_fall','cc_alcoholintoxication','cc_motorvehiclecrash','cc_dizziness','cc_cough','cc_alteredmentalstatus','cc_respiratorydistress','cc_strokealert','cc_cardiacarrest','cc_hypotension','cc_suicidal']
numeric_features = [c for c in base_numeric + cc_features if c in df.columns]
categorical_features = [c for c in base_categorical if c in df.columns]
features = numeric_features + categorical_features
X = df[features]
y = df['esi_numeric']
features

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y)

numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler(with_mean=False))])
categorical_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True))])
preprocess = ColumnTransformer([('num', numeric_pipe, numeric_features), ('cat', categorical_pipe, categorical_features)])

models = {
    'dummy_stratified': Pipeline([('preprocess', preprocess), ('model', DummyClassifier(strategy='stratified', random_state=RANDOM_SEED))]),
    'logistic_regression': Pipeline([('preprocess', preprocess), ('model', LogisticRegression(max_iter=200, class_weight='balanced', solver='lbfgs', n_jobs=-1, random_state=RANDOM_SEED))]),
    'decision_tree': Pipeline([('preprocess', preprocess), ('model', DecisionTreeClassifier(max_depth=TREE_MAX_DEPTH, min_samples_leaf=100, class_weight='balanced', random_state=RANDOM_SEED))]),
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    report = classification_report(y_test, pred, labels=[1,2,3,4,5], output_dict=True, zero_division=0)
    results[name] = {'accuracy': accuracy_score(y_test, pred), 'report': report, 'confusion_matrix': confusion_matrix(y_test, pred, labels=[1,2,3,4,5])}
    print(name, 'accuracy=', round(results[name]['accuracy'], 3), 'ESI 1 recall=', round(report['1']['recall'], 3), 'macro F1=', round(report['macro avg']['f1-score'], 3))

In [ ]:
pd.read_csv(ROOT / 'week-6' / 'assignments' / 'week6_model_summary.csv')

## Primary Metric

The primary metric is recall for ESI Level 1. In clinical language, the first safety question is: of the patients who truly needed the highest urgency level, how many did the model correctly identify as highest urgency? Overall accuracy can look acceptable on an imbalanced dataset while still missing rare critical cases.

## Output Files

The generated confusion matrices and summary images are stored in `week-6/outputs/`. The model result tables are stored in `week-6/assignments/`. The fitted baseline model artifacts are stored in `week-6/models/`.